# Lab 3 — Scenario Discovery: PRIM and Dimensional Stacking

## Background

**Scenario discovery** is an exploratory modelling technique for identifying the combinations
of uncertain conditions that lead to unacceptable outcomes ("failures"). Rather than asking
*what is the optimal policy?*, it asks:

> *Under which combinations of deeply uncertain conditions does this policy fail?*

This is a core step in **Robust Decision Making (RDM)**: by understanding when and why a
policy fails, decision-makers can either redesign the policy or prepare contingency plans.

### Patient Rule Induction Method (PRIM)

PRIM is a supervised subspace partitioning algorithm. Given a dataset of scenarios and a
binary failure label $y \in \{0, 1\}$, PRIM iteratively peels away scenarios that are
*not* of interest to find a tight, axis-aligned box that concentrates the cases of interest.

Two key statistics characterise a PRIM box:

$$\text{Coverage} = \frac{|\text{cases of interest in box}|}{|\text{all cases of interest}|}$$

$$\text{Density} = \frac{|\text{cases of interest in box}|}{|\text{all cases in box}|}$$

A good box has **high coverage** (captures most failures) and **high density** (most scenarios
inside the box are failures). These two objectives conflict — the analyst must choose a box
on the coverage–density trade-off curve.

### Dimensional Stacking

Dimensional stacking is a visual alternative to PRIM. The input space is discretised and
arranged as a nested grid of images. Coloured cells indicate the fraction of failures in
each region. It provides a global overview of the failure space without fitting a model.

### What you will do

1. Set up the lake model and generate 1 000 scenarios × 10 random policies.
2. Define a failure criterion (worst 10% reliability).
3. Apply PRIM: inspect the trade-off plot, select a box, and interpret its limits.
4. Visualise with dimensional stacking.
5. Increase sample size to 10 000 for better coverage.

## 1. Model setup and ensemble generation

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from lakemodel_function import lake_problem
from ema_workbench import (Model, RealParameter, ScalarOutcome,
                           MultiprocessingEvaluator, ema_logging)
from ema_workbench.analysis import prim, dimensional_stacking

ema_logging.log_to_stderr(ema_logging.INFO)

# --- Model definition (same as previous labs) ---
lake_model = Model('lakeproblem', function=lake_problem)
lake_model.time_horizon = 100

lake_model.uncertainties = [
    RealParameter('mean',  0.01,  0.05),
    RealParameter('stdev', 0.001, 0.005),
    RealParameter('b',     0.1,   0.45),
    RealParameter('q',     2.0,   4.5),
    RealParameter('delta', 0.93,  0.99),
]
lake_model.levers = [RealParameter(f'l{i}', 0, 0.1)
                     for i in range(lake_model.time_horizon)]
lake_model.outcomes = [
    ScalarOutcome('max_P'),
    ScalarOutcome('utility'),
    ScalarOutcome('inertia'),
    ScalarOutcome('reliability'),
]

In [ ]:
# Run 1 000 scenarios × 10 randomly sampled policies
n_scenarios = 1000
n_policies  = 10

with MultiprocessingEvaluator(lake_model, n_processes=-1) as evaluator:
    results = evaluator.perform_experiments(n_scenarios, n_policies)

experiments, outcomes = results
print(f'Experiments: {experiments.shape[0]} rows, {experiments.shape[1]} columns')
print(f'Unique policies: {experiments["policy"].nunique()}')

## 2. Prepare data for scenario discovery

The `experiments` DataFrame includes columns for both the uncertain parameters and the
100 decision levers (`l0` … `l99`). For scenario discovery, we are interested only in
the **uncertain parameters** — the levers define the policy and are not causes of failure
in the sense we are investigating here.

We drop the lever columns to keep only the five uncertainty dimensions.

In [ ]:
# Drop the 100 lever columns; keep only the 5 uncertainty parameters
lever_names = [l.name for l in lake_model.levers]
cleaned_experiments = experiments.drop(columns=lever_names)

print('Columns kept:', list(cleaned_experiments.columns))
print('Shape:', cleaned_experiments.shape)

## 3. Define the failure criterion

We define **failure** as belonging to the **worst 10% of reliability outcomes** across all
experiments. This is a percentile-based criterion: relative to the ensemble, these are the
scenarios where the lake's safety record is worst.

The binary vector `y` is `True` for failures and `False` for successes.

In [ ]:
reliability = outcomes['reliability']

# Threshold: 10th percentile of reliability
threshold = np.percentile(reliability, 10)
print(f'Reliability 10th percentile: {threshold:.4f}')

# Binary failure label
y = reliability < threshold
print(f'Number of failures: {y.sum()} out of {len(y)} ({100*y.mean():.1f}%)')

## 4. PRIM — Patient Rule Induction Method

We initialise PRIM with:
- `cleaned_experiments`: the 5-dimensional uncertainty space (all scenarios).
- `y`: the binary failure classification.
- `threshold=0.8`: we require at least 80% density (precision) in the final box.

`find_box()` runs the peeling algorithm and returns the best box.

In [ ]:
# Initialise and run PRIM
prim_alg = prim.Prim(cleaned_experiments, y)
box1 = prim_alg.find_box()

### 4.1 Coverage–density trade-off plot

Each point on the trade-off plot is a candidate PRIM box at a different peeling depth.
The analyst chooses a box that balances:
- **High coverage**: captures a large fraction of the failures.
- **High density**: most scenarios inside the box are failures.
- **Low complexity** (few restricted dimensions): interpretability.

A density below 0.5 means less than half the scenarios in the box are failures — generally
avoid such boxes. Look for the "elbow" in the curve.

In [ ]:
box1.show_tradeoff()
plt.show()

### 4.2 Inspect the selected box

`inspect(style='graph')` shows the limits of the selected box for each parameter.
- **Grey bars**: full range of each parameter across all scenarios.
- **Blue segment**: the restricted range in the PRIM box.
- **Numbers**: exact box limits.
- **Quasi p-values**: statistical significance of each restriction (lower → more significant).

Parameters without blue segments are not restricted (not important for this box).

In [ ]:
box1.inspect(style='graph')
plt.show()

## 5. Dimensional stacking

Dimensional stacking provides an independent visualisation of the failure space. It does
not fit a model — it shows raw failure rates in each region of the input space.

The axes are automatically ranked by importance (most important outer, least important inner).
White cells indicate missing data (no scenarios fell in that region).

In [ ]:
dimensional_stacking.create_pivot_plot(cleaned_experiments, y)
plt.suptitle('Dimensional stacking — worst 10% reliability', y=1.01)
plt.show()

### 5.1 Increase sample size for better coverage

White cells in the dimensional stacking plot indicate regions of the uncertainty space
that were not sampled. Running more scenarios (10 000) fills these gaps and gives
a more complete picture of the failure landscape.

In [ ]:
n_scenarios = 10000
n_policies  = 10

with MultiprocessingEvaluator(lake_model, n_processes=-1) as evaluator:
    results_large = evaluator.perform_experiments(n_scenarios, n_policies)

experiments_l, outcomes_l = results_large
cleaned_experiments_l = experiments_l.drop(
    columns=[l.name for l in lake_model.levers]
)

reliability_l = outcomes_l['reliability']
y_l = reliability_l < np.percentile(reliability_l, 10)

dimensional_stacking.create_pivot_plot(cleaned_experiments_l, y_l)
plt.suptitle('Dimensional stacking — 10 000 scenarios', y=1.01)
plt.show()

## Summary

Both PRIM and dimensional stacking converge on the same structural insight: **low `b`
(natural removal rate) and low `q` (recycling exponent)** are the conditions most associated
with poor reliability. When the lake removes phosphorus slowly (`b` low) and the recycling
dynamics are weaker (`q` low), even modest anthropogenic releases push the system toward
the eutrophication threshold.

This finding is consistent with the Sobol analysis in Lab 2, confirming that `b` and `q`
are the primary drivers of reliability risk.

In **Lab 5** you will move from exploring the uncertainty space to searching the lever space
with many-objective evolutionary algorithms.